# Projeto Integrador Interdisciplinar II — Terra Signal
## Arquitetura Medalhão e análise da carteira de clientes (Churn)

**Base:** `history.csv` — 5.634 registros e 23 variáveis.  
**Objetivo:** estruturar os dados nas camadas Bronze, Silver e Gold e investigar características associadas ao cancelamento de clientes da Terra Signal.

> **Como executar:** deixe este notebook e o arquivo `history.csv` na mesma pasta. Execute as células em ordem, de cima para baixo. O notebook preserva os dados originais na Bronze; a Silver contém os tratamentos documentados; a Gold contém indicadores e tabelas preparadas para as análises.

### Perguntas de negócio
1. Qual é a proporção de clientes que cancelaram e quais características de perfil aparecem com maior frequência nesse grupo?
2. Como a taxa de cancelamento varia conforme o contrato, o tempo de relacionamento e os serviços contratados?
3. Existe associação entre mensalidade, renda mensal, comprometimento da renda e cancelamento?
4. Quais métodos de pagamento e modalidades de internet apresentam maiores taxas de cancelamento?
5. Quais ações a Terra Signal pode investigar ou testar para reduzir o churn?

### Arquitetura Medalhão
| Camada | Conteúdo | Regra deste projeto |
|---|---|---|
| **Bronze** | Dados recebidos de `history.csv` | Leitura e diagnóstico. Não alterar os dados originais. |
| **Silver** | Dados limpos e padronizados | Corrigir tipos e categorias; tratar somente situações justificáveis; documentar ausências remanescentes. |
| **Gold** | Dados analíticos | Criar indicadores e tabelas agregadas para responder às perguntas de negócio. |

A análise é exploratória. Diferenças entre grupos indicam **associações**, não comprovam que uma variável cause o cancelamento.

## 1. Importações e configurações

Bibliotecas utilizadas: `pandas` e `numpy` para manipulação; `plotly` para gráficos interativos; `matplotlib` e `wordcloud` para a análise complementar dos comentários.

Caso alguma biblioteca não esteja instalada no ambiente, execute no terminal: `pip install pandas numpy plotly matplotlib wordcloud nbformat`.

In [4]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

# Arquivo esperado na mesma pasta do notebook.
CAMINHO_CSV = Path("history.csv")
if not CAMINHO_CSV.exists():
    candidatos = [Path.cwd() / "history.csv", Path.cwd().parent / "history.csv"]
    CAMINHO_CSV = next((p for p in candidatos if p.exists()), CAMINHO_CSV)

if not CAMINHO_CSV.exists():
    raise FileNotFoundError(
        "Não encontrei history.csv. Coloque o CSV na mesma pasta deste notebook "
        "ou ajuste a variável CAMINHO_CSV."
    )

print(f"Arquivo localizado: {CAMINHO_CSV.resolve()}")

Arquivo localizado: C:\Users\giova\OneDrive\Documentos\PII2_databricks\history.csv


## 2. Camada Bronze — leitura e diagnóstico dos dados originais

A Bronze é uma cópia fiel do arquivo de origem. Nesta etapa, não renomeamos, convertemos nem preenchemos valores. O objetivo é entender a estrutura e registrar os problemas antes do tratamento.

In [6]:
# Leitura do CSV original. O arquivo utiliza vírgula como delimitador.
df_bronze = pd.read_csv(CAMINHO_CSV, encoding="utf-8", sep=",")

print("Dimensão da camada Bronze:", df_bronze.shape)
display(df_bronze.head())
display(df_bronze.tail())

Dimensão da camada Bronze: (5634, 23)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome
0,4578-PHJYZ,Male,0,Yes,Yes,52,Yes,No,DSL,No,Yes,Yes,Yes,Yes,No,One year,Yes,Electronic check,68.75,"3,482.85",No,I have been a customer with this internet provider for over four years now and I have been satis...,6532
1,6289-CPNLD,Male,0,Yes,Yes,33,Yes,No,DSL,No,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,73.90,"2,405.05",Yes,"I've been a customer with this company for over 2 and a half years now, and unfortunately I have...",7634
2,2682-KEVRP,Female,1,No,No,22,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,20.05,417.00,No,I have been a customer with this company for 22 months now and I have been satisfied with the se...,3628
3,5697-GOMBF,Female,1,Yes,Yes,28,No,No phone service,DSL,No,No,No,No,Yes,No,Month-to-month,Yes,Electronic check,35.90,973.65,No,I have been a customer with this internet provider for 28 months now and I have been very satisf...,7851
4,9717-QEBGU,Male,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.30,144.00,No,I have been using the fiber optic internet service for the past 2 months and I am very satisfied...,1691


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome
5629,8909-BOLNL,Male,0,No,No,3,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Mailed check,21.20,52.05,No,I have been a customer with this company for 3 months now and I am pleased to say that I have ha...,8354
5630,2832-KJCRD,Female,0,No,No,38,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,103.65,"3,988.50",No,I have been a customer with this company for 38 months now and I have had a great experience so ...,9308
5631,7975-JMZNT,Male,0,Yes,No,66,Yes,Yes,DSL,Yes,Yes,Yes,Yes,Yes,Yes,Two year,No,Bank transfer (automatic),91.70,"6,075.90",No,"I have been a customer with this DSL internet service provider for 66 months now, and I must say...",2113
5632,4250-ZBWLV,Male,0,No,No,68,Yes,Yes,Fiber optic,No,Yes,Yes,Yes,Yes,Yes,One year,No,Electronic check,108.45,"7,176.55",Yes,I have been a customer with this company for over 5 years and have always enjoyed their fiber op...,3904
5633,3915-ODIYG,Male,1,No,No,60,Yes,Yes,DSL,Yes,Yes,No,No,No,Yes,One year,Yes,Electronic check,71.75,"4,374.55",No,I have been a loyal customer of this internet service provider for 5 years now and I have been v...,3016


In [7]:
# Estrutura e tipos originalmente identificados pelo pandas.
print("Tipos de dados na Bronze:")
display(df_bronze.dtypes.rename("tipo_original").to_frame())

print("Resumo da estrutura:")
df_bronze.info()

Tipos de dados na Bronze:


,tipo_original
customerID,str
gender,str
SeniorCitizen,int64
Partner,str
Dependents,str
tenure,str
PhoneService,str
MultipleLines,str
InternetService,str
OnlineSecurity,str


Resumo da estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 5634 entries, 0 to 5633
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5634 non-null   str    
 1   gender            5634 non-null   str    
 2   SeniorCitizen     5634 non-null   int64  
 3   Partner           5634 non-null   str    
 4   Dependents        5634 non-null   str    
 5   tenure            5634 non-null   str    
 6   PhoneService      5634 non-null   str    
 7   MultipleLines     5634 non-null   str    
 8   InternetService   5634 non-null   str    
 9   OnlineSecurity    5634 non-null   str    
 10  OnlineBackup      5634 non-null   str    
 11  DeviceProtection  5634 non-null   str    
 12  TechSupport       5634 non-null   str    
 13  StreamingTV       5634 non-null   str    
 14  StreamingMovies   5634 non-null   str    
 15  Contract          5634 non-null   str    
 16  PaperlessBilling  5634 non-null 

In [8]:
# Diagnóstico de valores ausentes e duplicidades na Bronze.
resumo_nulos_bronze = (
    df_bronze.isna().sum().rename("quantidade_ausente").to_frame()
)
resumo_nulos_bronze["percentual_ausente"] = (
    resumo_nulos_bronze["quantidade_ausente"] / len(df_bronze) * 100
)
display(resumo_nulos_bronze.sort_values("quantidade_ausente", ascending=False))

print("Linhas totalmente duplicadas:", df_bronze.duplicated().sum())
print("customerID duplicados:", df_bronze["customerID"].duplicated().sum())

,quantidade_ausente,percentual_ausente
TotalCharges,278,4.93
gender,0,0.00
customerID,0,0.00
Partner,0,0.00
Dependents,0,0.00
tenure,0,0.00
SeniorCitizen,0,0.00
PhoneService,0,0.00
MultipleLines,0,0.00
OnlineSecurity,0,0.00


Linhas totalmente duplicadas: 0
customerID duplicados: 0


In [9]:
# Verificação inicial das categorias e de valores suspeitos.
colunas_categoricas_bronze = df_bronze.select_dtypes(include="object").columns.tolist()
for coluna in colunas_categoricas_bronze:
    if coluna != "CustomerFeedback":
        print(f"\\n{coluna} — valores encontrados:")
        print(df_bronze[coluna].value_counts(dropna=False).to_dict())

print("\\nValores originais de tenure:")
print(df_bronze["tenure"].value_counts(dropna=False).tail(10))

\ncustomerID — valores encontrados:
{'4578-PHJYZ': 1, '6289-CPNLD': 1, '2682-KEVRP': 1, '5697-GOMBF': 1, '9717-QEBGU': 1, '6797-LNAQX': 1, '3050-GBUSH': 1, '8020-BWHYL': 1, '6752-APNJL': 1, '1760-CAZHT': 1, '3070-FNFZQ': 1, '4452-QIIEB': 1, '7767-UXAGJ': 1, '9793-WECQC': 1, '7029-IJEJK': 1, '5419-JKZNQ': 1, '3351-NQLDI': 1, '1335-HQMKX': 1, '4702-IOQDC': 1, '2043-WVTQJ': 1, '5320-BRKGK': 1, '5945-AZYHT': 1, '6653-CBBOM': 1, '7995-ZHHNZ': 1, '8267-ZNYVZ': 1, '0093-EXYQL': 1, '7717-BICXI': 1, '2323-ARSVR': 1, '5281-BUZGT': 1, '1930-WNXSB': 1, '0612-RTZZA': 1, '4983-CCWMC': 1, '4817-QRJSX': 1, '8775-CEBBJ': 1, '2154-KVJFF': 1, '8749-CLJXC': 1, '5028-HTLJB': 1, '0661-KBKPA': 1, '9063-ZGTUY': 1, '2419-FSORS': 1, '4587-NUKOX': 1, '6770-UAYGJ': 1, '3537-RYBHH': 1, '9720-JJJOR': 1, '1984-GPTEH': 1, '8809-RIHDD': 1, '0130-SXOUN': 1, '1320-REHCS': 1, '1150-WFARN': 1, '6996-KNSML': 1, '7576-OYWBN': 1, '5789-LDFXO': 1, '1414-YADCW': 1, '9743-DQKQW': 1, '9537-JALFH': 1, '7446-YPODE': 1, '4803-LBYPN

C:\Users\giova\AppData\Local\Temp\ipykernel_31560\2620343090.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_categoricas_bronze = df_bronze.select_dtypes(include="object").columns.tolist()


### 2.1 Dicionário de dados — base recebida

As unidades monetárias não estão especificadas explicitamente no material da base. Por isso, os campos financeiros são descritos como **unidades monetárias da base**; antes de apresentar valores como reais ou outra moeda, confirmem essa informação com a documentação do conjunto de dados.

In [10]:
dicionario = pd.DataFrame([
    ("customerID", "Identificador único do cliente", "Qualitativa nominal", "Código", "string"),
    ("gender", "Gênero informado no cadastro", "Qualitativa nominal", "Male / Female", "category"),
    ("SeniorCitizen", "Indicador de cliente idoso", "Quantitativa discreta / binária", "0 = não; 1 = sim", "Int64"),
    ("Partner", "Indica se possui parceiro(a)", "Qualitativa nominal", "Yes / No", "category"),
    ("Dependents", "Indica se possui dependentes", "Qualitativa nominal", "Yes / No", "category"),
    ("tenure", "Tempo de relacionamento com a empresa", "Quantitativa discreta", "Meses (0 a 72); pode conter unknown", "Int64"),
    ("PhoneService", "Indica contratação de serviço telefônico", "Qualitativa nominal", "Yes / No", "category"),
    ("MultipleLines", "Indica contratação de múltiplas linhas", "Qualitativa nominal", "Yes / No / No phone service", "category"),
    ("InternetService", "Tipo de serviço de internet contratado", "Qualitativa nominal", "DSL / Fiber optic / No", "category"),
    ("OnlineSecurity", "Indica contratação de segurança online", "Qualitativa nominal", "Yes / No / No internet service", "category"),
    ("OnlineBackup", "Indica contratação de backup online", "Qualitativa nominal", "Yes / No / No internet service", "category"),
    ("DeviceProtection", "Indica contratação de proteção de dispositivo", "Qualitativa nominal", "Yes / No / No internet service", "category"),
    ("TechSupport", "Indica contratação de suporte técnico", "Qualitativa nominal", "Yes / No / No internet service", "category"),
    ("StreamingTV", "Indica contratação de streaming de TV", "Qualitativa nominal", "Yes / No / No internet service", "category"),
    ("StreamingMovies", "Indica contratação de streaming de filmes", "Qualitativa nominal", "Yes / No / No internet service", "category"),
    ("Contract", "Modalidade de contrato", "Qualitativa nominal", "Month-to-month / One year / Two year", "category"),
    ("PaperlessBilling", "Indica uso de fatura eletrônica", "Qualitativa nominal", "Yes / No", "category"),
    ("PaymentMethod", "Método de pagamento", "Qualitativa nominal", "Electronic check / Mailed check / Credit card / Bank transfer", "category"),
    ("MonthlyCharges", "Valor cobrado mensalmente", "Quantitativa contínua", "Unidade monetária da base por mês", "float64"),
    ("TotalCharges", "Valor acumulado de cobranças", "Quantitativa contínua", "Unidade monetária da base; acumulado", "Float64"),
    ("Churn", "Indica se o cliente cancelou", "Qualitativa nominal", "Yes / No", "category"),
    ("CustomerFeedback", "Comentário textual sobre a experiência", "Qualitativa textual", "Texto livre", "string"),
    ("MonthlyIncome", "Renda mensal registrada", "Quantitativa discreta", "Unidade monetária da base por mês; confirmar moeda", "Int64"),
], columns=["variavel_original", "descricao", "classificacao", "valores_unidade", "tipo_recomendado"])

display(dicionario)
print("Quantidade de variáveis documentadas:", len(dicionario))

,variavel_original,descricao,classificacao,valores_unidade,tipo_recomendado
0,customerID,Identificador único do cliente,Qualitativa nominal,Código,string
1,gender,Gênero informado no cadastro,Qualitativa nominal,Male / Female,category
2,SeniorCitizen,Indicador de cliente idoso,Quantitativa discreta / binária,0 = não; 1 = sim,Int64
3,Partner,Indica se possui parceiro(a),Qualitativa nominal,Yes / No,category
4,Dependents,Indica se possui dependentes,Qualitativa nominal,Yes / No,category
5,tenure,Tempo de relacionamento com a empresa,Quantitativa discreta,Meses (0 a 72); pode conter unknown,Int64
6,PhoneService,Indica contratação de serviço telefônico,Qualitativa nominal,Yes / No,category
7,MultipleLines,Indica contratação de múltiplas linhas,Qualitativa nominal,Yes / No / No phone service,category
8,InternetService,Tipo de serviço de internet contratado,Qualitativa nominal,DSL / Fiber optic / No,category
9,OnlineSecurity,Indica contratação de segurança online,Qualitativa nominal,Yes / No / No internet service,category


Quantidade de variáveis documentadas: 23


### 2.2 Diagnóstico inicial — problemas que serão tratados

A inspeção da base original identifica:
- `tenure` contém o texto `unknown` em 19 registros, embora represente meses e deva ser numérico.
- `TotalCharges` possui 278 valores ausentes.
- `PhoneService` apresenta duas categorias com letras minúsculas (`yes` e `no`), além das formas padronizadas (`Yes` e `No`).
- A base não apresenta linhas totalmente duplicadas nem identificadores `customerID` duplicados.

Os demais valores ausentes, as categorias e as regras de coerência serão conferidos novamente na Silver.

## 3. Camada Silver — limpeza, conversão e padronização

A Silver será criada a partir de `df_bronze.copy()`. A Bronze permanece intacta.

### Decisões de tratamento
1. Renomear as 23 colunas para nomes em português, preservando o identificador `id_cliente`.
2. Converter `DuracaoContrato` para inteiro anulável (`Int64`), transformando `unknown` em valor ausente. Não estimar a duração a partir da divisão entre cobrança acumulada e mensalidade.
3. Converter as variáveis financeiras para tipos numéricos. `CobrancaTotal` utilizará `Float64`, que aceita valores ausentes.
4. Padronizar `ServicoTelefonico` (`yes`/`no` para `Yes`/`No`) e remover espaços periféricos das categorias.
5. Preencher com zero somente os 10 registros em que `DuracaoContrato = 0` e `CobrancaTotal` está ausente. A regra é: com zero meses de relacionamento, não há cobrança acumulada registrada. Os demais valores ausentes de `CobrancaTotal` serão preservados, pois não é possível reconstruir cobranças históricas apenas multiplicando mensalidade atual por duração.
6. Não excluir registros: não foram identificadas duplicidades de linha ou de identificador.

In [11]:
# Mapeamento dos nomes originais para nomes em português.
mapa_colunas = {
    "customerID": "id_cliente",
    "gender": "Genero",
    "SeniorCitizen": "Senioridade",
    "Partner": "Parceiro",
    "Dependents": "Dependentes",
    "tenure": "DuracaoContrato",
    "PhoneService": "ServicoTelefonico",
    "MultipleLines": "MultiplasLinhas",
    "InternetService": "ServicoInternet",
    "OnlineSecurity": "SegurancaOnline",
    "OnlineBackup": "BackupOnline",
    "DeviceProtection": "ProtecaoDispositivo",
    "TechSupport": "SuporteTecnico",
    "StreamingTV": "StreamingTV",
    "StreamingMovies": "StreamingFilmes",
    "Contract": "Contrato",
    "PaperlessBilling": "FaturaEletronica",
    "PaymentMethod": "MetodoPagamento",
    "MonthlyCharges": "CobrancaMensal",
    "TotalCharges": "CobrancaTotal",
    "Churn": "Cancelamento",
    "CustomerFeedback": "FeedbackCliente",
    "MonthlyIncome": "RendaMensal",
}

df_silver = df_bronze.copy().rename(columns=mapa_colunas)
print("Dimensão inicial da Silver:", df_silver.shape)

Dimensão inicial da Silver: (5634, 23)


In [12]:
# Colunas categóricas: retirar apenas espaços no início/fim, sem alterar o significado.
colunas_categoria = [
    "Genero", "Parceiro", "Dependentes", "ServicoTelefonico", "MultiplasLinhas",
    "ServicoInternet", "SegurancaOnline", "BackupOnline", "ProtecaoDispositivo",
    "SuporteTecnico", "StreamingTV", "StreamingFilmes", "Contrato",
    "FaturaEletronica", "MetodoPagamento", "Cancelamento"
]

for coluna in colunas_categoria:
    df_silver[coluna] = df_silver[coluna].astype("string").str.strip()

# Padronização de Yes/No sem modificar categorias especiais como 'No phone service'.
def padronizar_yes_no(serie):
    valores = serie.astype("string").str.strip()
    mapa = {"yes": "Yes", "no": "No"}
    return valores.map(lambda valor: mapa.get(valor.casefold(), valor) if pd.notna(valor) else pd.NA)

colunas_sim_nao = [
    "Genero", "Parceiro", "Dependentes", "ServicoTelefonico", "MultiplasLinhas",
    "SegurancaOnline", "BackupOnline", "ProtecaoDispositivo", "SuporteTecnico",
    "StreamingTV", "StreamingFilmes", "FaturaEletronica", "Cancelamento"
]
for coluna in colunas_sim_nao:
    df_silver[coluna] = padronizar_yes_no(df_silver[coluna])

# tenure: 'unknown' deixa de ser texto e passa a ser ausente explícito.
duracao_texto = df_silver["DuracaoContrato"].astype("string").str.strip()
duracao_texto = duracao_texto.mask(duracao_texto.str.casefold().eq("unknown"), pd.NA)
df_silver["DuracaoContrato"] = pd.to_numeric(duracao_texto, errors="coerce").astype("Int64")

# Conversão de colunas numéricas.
df_silver["Senioridade"] = pd.to_numeric(df_silver["Senioridade"], errors="coerce").astype("Int64")
df_silver["CobrancaMensal"] = pd.to_numeric(df_silver["CobrancaMensal"], errors="coerce").astype("float64")
df_silver["CobrancaTotal"] = pd.to_numeric(df_silver["CobrancaTotal"], errors="coerce").astype("Float64")
df_silver["RendaMensal"] = pd.to_numeric(df_silver["RendaMensal"], errors="coerce").astype("Int64")

# Tipos textuais e categorias.
df_silver["id_cliente"] = df_silver["id_cliente"].astype("string")
df_silver["FeedbackCliente"] = df_silver["FeedbackCliente"].astype("string")
for coluna in colunas_categoria:
    df_silver[coluna] = df_silver[coluna].astype("category")

# Regra de negócio documentada: tenure igual a zero e total ausente -> total acumulado zero.
mascara_zero_meses = df_silver["DuracaoContrato"].eq(0) & df_silver["CobrancaTotal"].isna()
quantidade_preenchida_zero = int(mascara_zero_meses.sum())
df_silver.loc[mascara_zero_meses, "CobrancaTotal"] = 0.0

print(f"Valores de CobrancaTotal preenchidos com zero pela regra tenure=0: {quantidade_preenchida_zero}")
print("Dimensão final da Silver:", df_silver.shape)
display(df_silver.head())

Valores de CobrancaTotal preenchidos com zero pela regra tenure=0: 10
Dimensão final da Silver: (5634, 23)


,id_cliente,Genero,Senioridade,Parceiro,Dependentes,DuracaoContrato,ServicoTelefonico,MultiplasLinhas,ServicoInternet,SegurancaOnline,BackupOnline,ProtecaoDispositivo,SuporteTecnico,StreamingTV,StreamingFilmes,Contrato,FaturaEletronica,MetodoPagamento,CobrancaMensal,CobrancaTotal,Cancelamento,FeedbackCliente,RendaMensal
0,4578-PHJYZ,Male,0,Yes,Yes,52,Yes,No,DSL,No,Yes,Yes,Yes,Yes,No,One year,Yes,Electronic check,68.75,"3,482.85",No,I have been a customer with this internet provider for over four years now and I have been satis...,6532
1,6289-CPNLD,Male,0,Yes,Yes,33,Yes,No,DSL,No,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,73.90,"2,405.05",Yes,"I've been a customer with this company for over 2 and a half years now, and unfortunately I have...",7634
2,2682-KEVRP,Female,1,No,No,22,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,20.05,417.00,No,I have been a customer with this company for 22 months now and I have been satisfied with the se...,3628
3,5697-GOMBF,Female,1,Yes,Yes,28,No,No phone service,DSL,No,No,No,No,Yes,No,Month-to-month,Yes,Electronic check,35.90,973.65,No,I have been a customer with this internet provider for 28 months now and I have been very satisf...,7851
4,9717-QEBGU,Male,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.30,144.00,No,I have been using the fiber optic internet service for the past 2 months and I am very satisfied...,1691


In [13]:
# Conferência dos tipos após o tratamento.
display(df_silver.dtypes.rename("tipo_silver").to_frame())
print("Distribuição dos tipos na Silver:")
display(df_silver.dtypes.astype(str).value_counts().rename_axis("tipo").to_frame("quantidade_colunas"))

,tipo_silver
id_cliente,string
Genero,category
Senioridade,Int64
Parceiro,category
Dependentes,category
DuracaoContrato,Int64
ServicoTelefonico,category
MultiplasLinhas,category
ServicoInternet,category
SegurancaOnline,category


Distribuição dos tipos na Silver:


,quantidade_colunas
tipo,
category,16
Int64,3
string,2
float64,1
Float64,1


### 3.1 Verificações de qualidade e regras de negócio

As verificações abaixo avaliam limites, integridade e relações entre os serviços. Valores ausentes mantidos por falta de informação não são tratados como inconsistências automaticamente.

In [14]:
# Valores fora dos intervalos esperados e categorias válidas.
regras_validacao = {
    "DuracaoContrato fora de 0 a 72": int((df_silver["DuracaoContrato"].notna() & ~df_silver["DuracaoContrato"].between(0, 72)).sum()),
    "CobrancaMensal negativa": int((df_silver["CobrancaMensal"] < 0).sum()),
    "CobrancaTotal negativa": int((df_silver["CobrancaTotal"].dropna() < 0).sum()),
    "RendaMensal menor ou igual a zero": int((df_silver["RendaMensal"].dropna() <= 0).sum()),
    "Senioridade fora de 0/1": int((df_silver["Senioridade"].notna() & ~df_silver["Senioridade"].isin([0, 1])).sum()),
    "Cancelamento diferente de Yes/No": int((~df_silver["Cancelamento"].isin(["Yes", "No"])).sum()),
    "ID de cliente duplicado": int(df_silver["id_cliente"].duplicated().sum()),
    "Linhas totalmente duplicadas": int(df_silver.duplicated().sum()),
}

# Coerência entre PhoneService e MultipleLines.
mascara_sem_telefone = df_silver["ServicoTelefonico"].eq("No")
regras_validacao["Sem telefone mas MultipleLines diferente de 'No phone service'"] = int(
    (mascara_sem_telefone & df_silver["MultiplasLinhas"].ne("No phone service")).sum()
)

# Coerência dos seis serviços adicionais de internet.
colunas_sem_internet = ["SegurancaOnline", "BackupOnline", "ProtecaoDispositivo", "SuporteTecnico", "StreamingTV", "StreamingFilmes"]
mascara_sem_internet = df_silver["ServicoInternet"].eq("No")
for coluna in colunas_sem_internet:
    regras_validacao[f"Sem internet mas {coluna} diferente de 'No internet service'"] = int(
        (mascara_sem_internet & df_silver[coluna].ne("No internet service")).sum()
    )

validacao_regras = pd.DataFrame.from_dict(regras_validacao, orient="index", columns=["quantidade_violacoes"])
display(validacao_regras)

,quantidade_violacoes
DuracaoContrato fora de 0 a 72,0
CobrancaMensal negativa,0
CobrancaTotal negativa,0
RendaMensal menor ou igual a zero,0
Senioridade fora de 0/1,0
Cancelamento diferente de Yes/No,0
ID de cliente duplicado,0
Linhas totalmente duplicadas,0
Sem telefone mas MultipleLines diferente de 'No phone service',0
Sem internet mas SegurancaOnline diferente de 'No internet service',0


In [15]:
# Valores únicos depois da padronização.
for coluna in ["ServicoTelefonico", "MultiplasLinhas", "ServicoInternet", "Contrato", "MetodoPagamento", "Cancelamento"]:
    print(f"\\n{coluna}:")
    print(df_silver[coluna].value_counts(dropna=False).to_dict())

print("\\nAusentes por variável na Silver:")
display(df_silver.isna().sum().sort_values(ascending=False).rename("ausentes").to_frame())

\nServicoTelefonico:
{'Yes': 5071, 'No': 563}
\nMultiplasLinhas:
{'No': 2704, 'Yes': 2367, 'No phone service': 563}
\nServicoInternet:
{'Fiber optic': 2477, 'DSL': 1924, 'No': 1233}
\nContrato:
{'Month-to-month': 3099, 'Two year': 1347, 'One year': 1188}
\nMetodoPagamento:
{'Electronic check': 1888, 'Mailed check': 1283, 'Credit card (automatic)': 1234, 'Bank transfer (automatic)': 1229}
\nCancelamento:
{'No': 4130, 'Yes': 1504}
\nAusentes por variável na Silver:


,ausentes
CobrancaTotal,268
DuracaoContrato,19
Genero,0
Senioridade,0
Parceiro,0
Dependentes,0
id_cliente,0
ServicoTelefonico,0
MultiplasLinhas,0
SegurancaOnline,0


### 3.2 Comparação da qualidade — Bronze × Silver

A comparação diferencia ausência real de um marcador textual desconhecido. Na Bronze, os 19 valores `unknown` de `tenure` não são nulos; na Silver, tornam-se ausências explícitas. Assim, o total de `NA` pode aumentar sem significar perda de informação: a Silver passa a representar corretamente que a duração não foi informada.

In [16]:
# Métricas específicas para comparar o antes e o depois.
phone_bronze = df_bronze["PhoneService"].astype("string").str.strip()
phone_silver = df_silver["ServicoTelefonico"].astype("string").str.strip()
tenure_bronze_text = df_bronze["tenure"].astype("string").str.strip().str.casefold()

comparacao_qualidade = pd.DataFrame({
    "Indicador": [
        "Quantidade de linhas",
        "Quantidade de colunas",
        "Linhas totalmente duplicadas",
        "IDs de cliente duplicados",
        "tenure ausente ou unknown",
        "TotalCharges / CobrancaTotal ausente",
        "PhoneService com escrita não padronizada",
    ],
    "Bronze": [
        len(df_bronze), df_bronze.shape[1], int(df_bronze.duplicated().sum()),
        int(df_bronze["customerID"].duplicated().sum()),
        int(df_bronze["tenure"].isna().sum() + tenure_bronze_text.eq("unknown").sum()),
        int(df_bronze["TotalCharges"].isna().sum()),
        int(phone_bronze.isin(["yes", "no"]).sum()),
    ],
    "Silver": [
        len(df_silver), df_silver.shape[1], int(df_silver.duplicated().sum()),
        int(df_silver["id_cliente"].duplicated().sum()),
        int(df_silver["DuracaoContrato"].isna().sum()),
        int(df_silver["CobrancaTotal"].isna().sum()),
        int(phone_silver.isin(["yes", "no"]).sum()),
    ]
})
comparacao_qualidade["Variação (Silver - Bronze)"] = comparacao_qualidade["Silver"] - comparacao_qualidade["Bronze"]
display(comparacao_qualidade)

,Indicador,Bronze,Silver,Variação (Silver - Bronze)
0,Quantidade de linhas,5634,5634,0
1,Quantidade de colunas,23,23,0
2,Linhas totalmente duplicadas,0,0,0
3,IDs de cliente duplicados,0,0,0
4,tenure ausente ou unknown,19,19,0
5,TotalCharges / CobrancaTotal ausente,278,268,-10
6,PhoneService com escrita não padronizada,70,0,-70


### 3.3 Dicionário de dados atualizado — Silver

In [17]:
traducao_dicionario = {
    "customerID": "id_cliente", "gender": "Genero", "SeniorCitizen": "Senioridade",
    "Partner": "Parceiro", "Dependents": "Dependentes", "tenure": "DuracaoContrato",
    "PhoneService": "ServicoTelefonico", "MultipleLines": "MultiplasLinhas",
    "InternetService": "ServicoInternet", "OnlineSecurity": "SegurancaOnline",
    "OnlineBackup": "BackupOnline", "DeviceProtection": "ProtecaoDispositivo",
    "TechSupport": "SuporteTecnico", "StreamingTV": "StreamingTV",
    "StreamingMovies": "StreamingFilmes", "Contract": "Contrato",
    "PaperlessBilling": "FaturaEletronica", "PaymentMethod": "MetodoPagamento",
    "MonthlyCharges": "CobrancaMensal", "TotalCharges": "CobrancaTotal",
    "Churn": "Cancelamento", "CustomerFeedback": "FeedbackCliente",
    "MonthlyIncome": "RendaMensal"
}
dicionario_silver = dicionario.copy()
dicionario_silver["coluna_silver"] = dicionario_silver["variavel_original"].map(traducao_dicionario)
dicionario_silver["tipo_observado_silver"] = dicionario_silver["coluna_silver"].map(lambda c: str(df_silver[c].dtype))
dicionario_silver = dicionario_silver[["variavel_original", "coluna_silver", "descricao", "classificacao", "valores_unidade", "tipo_observado_silver"]]
display(dicionario_silver)

,variavel_original,coluna_silver,descricao,classificacao,valores_unidade,tipo_observado_silver
0,customerID,id_cliente,Identificador único do cliente,Qualitativa nominal,Código,string
1,gender,Genero,Gênero informado no cadastro,Qualitativa nominal,Male / Female,category
2,SeniorCitizen,Senioridade,Indicador de cliente idoso,Quantitativa discreta / binária,0 = não; 1 = sim,Int64
3,Partner,Parceiro,Indica se possui parceiro(a),Qualitativa nominal,Yes / No,category
4,Dependents,Dependentes,Indica se possui dependentes,Qualitativa nominal,Yes / No,category
5,tenure,DuracaoContrato,Tempo de relacionamento com a empresa,Quantitativa discreta,Meses (0 a 72); pode conter unknown,Int64
6,PhoneService,ServicoTelefonico,Indica contratação de serviço telefônico,Qualitativa nominal,Yes / No,category
7,MultipleLines,MultiplasLinhas,Indica contratação de múltiplas linhas,Qualitativa nominal,Yes / No / No phone service,category
8,InternetService,ServicoInternet,Tipo de serviço de internet contratado,Qualitativa nominal,DSL / Fiber optic / No,category
9,OnlineSecurity,SegurancaOnline,Indica contratação de segurança online,Qualitativa nominal,Yes / No / No internet service,category


### 3.4 Armazenamento e conferência da Silver

O arquivo original permanece preservado. A Silver é exportada em um novo arquivo. Como CSV não armazena os tipos `category` e `Int64`, eles devem ser reaplicados quando o CSV for carregado novamente.

In [18]:
CAMINHO_SILVER = Path("base_silver.csv")
df_silver.to_csv(CAMINHO_SILVER, index=False, encoding="utf-8-sig")

# Leitura de conferência do arquivo exportado.
df_silver_conferencia = pd.read_csv(CAMINHO_SILVER, encoding="utf-8-sig")
print("Arquivo Silver salvo em:", CAMINHO_SILVER.resolve())
print("Dimensão após reabrir o CSV:", df_silver_conferencia.shape)
print("Mesma quantidade de linhas:", len(df_silver_conferencia) == len(df_silver))
print("Mesma quantidade de colunas:", df_silver_conferencia.shape[1] == df_silver.shape[1])

Arquivo Silver salvo em: C:\Users\giova\OneDrive\Documentos\PII2_databricks\base_silver.csv
Dimensão após reabrir o CSV: (5634, 23)
Mesma quantidade de linhas: True
Mesma quantidade de colunas: True


## 4. Camada Gold — preparação dos indicadores

A Gold parte exclusivamente da Silver. Nesta camada serão criadas variáveis derivadas e tabelas agregadas para responder às perguntas de negócio. A base de clientes continuará com um registro por cliente.

**Indicadores derivados:**
- `CancelamentoBinario`: 1 para cancelou e 0 para não cancelou.
- `FaixaPermanencia`: faixas de 12 meses para facilitar a leitura do tempo de relacionamento.
- `ComprometimentoRendaPct`: cobrança mensal dividida pela renda mensal, multiplicada por 100.
- `QuartilComprometimento`: quatro grupos de tamanho semelhante, do menor para o maior comprometimento.
- `QuantidadeServicos`: contagem dos serviços e recursos marcados como ativos.

O indicador de comprometimento só é uma aproximação útil se `MonthlyIncome` e `MonthlyCharges` estiverem na mesma moeda e periodicidade. Essa hipótese precisa ser confirmada na documentação da base antes de uma conclusão financeira definitiva.

In [19]:
df_gold = df_silver.copy()

# Indicador binário para facilitar cálculos de taxa.
df_gold["CancelamentoBinario"] = df_gold["Cancelamento"].map({"No": 0, "Yes": 1}).astype("Int64")

# Faixas de permanência: 0-12, 13-24, ..., 61-72 meses.
faixas_tenure = ["0–12 meses", "13–24 meses", "25–36 meses", "37–48 meses", "49–60 meses", "61–72 meses"]
df_gold["FaixaPermanencia"] = pd.cut(
    df_gold["DuracaoContrato"].astype("float64"),
    bins=[-1, 12, 24, 36, 48, 60, 72],
    labels=faixas_tenure,
    include_lowest=True
)

# Comprometimento da renda. Rendas nulas ou não positivas geram NA.
mascara_renda_valida = df_gold["RendaMensal"].notna() & df_gold["RendaMensal"].gt(0)
df_gold["ComprometimentoRendaPct"] = np.where(
    mascara_renda_valida,
    df_gold["CobrancaMensal"] / df_gold["RendaMensal"] * 100,
    np.nan
)

labels_quartis = ["Q1 — menor", "Q2", "Q3", "Q4 — maior"]
df_gold["QuartilComprometimento"] = pd.qcut(
    df_gold["ComprometimentoRendaPct"], q=4, labels=labels_quartis
)

# Contagem de recursos/serviços ativos. DSL e Fiber optic contam como internet ativa.
mapa_servicos = {
    "Telefone": df_gold["ServicoTelefonico"].eq("Yes"),
    "Múltiplas linhas": df_gold["MultiplasLinhas"].eq("Yes"),
    "Internet": df_gold["ServicoInternet"].isin(["DSL", "Fiber optic"]),
    "Segurança online": df_gold["SegurancaOnline"].eq("Yes"),
    "Backup online": df_gold["BackupOnline"].eq("Yes"),
    "Proteção do dispositivo": df_gold["ProtecaoDispositivo"].eq("Yes"),
    "Suporte técnico": df_gold["SuporteTecnico"].eq("Yes"),
    "Streaming de TV": df_gold["StreamingTV"].eq("Yes"),
    "Streaming de filmes": df_gold["StreamingFilmes"].eq("Yes"),
}
df_gold["QuantidadeServicos"] = pd.DataFrame(mapa_servicos, index=df_gold.index).sum(axis=1).astype("Int64")

print("Dimensão da Gold:", df_gold.shape)
display(df_gold[["id_cliente", "DuracaoContrato", "FaixaPermanencia", "CobrancaMensal", "RendaMensal", "ComprometimentoRendaPct", "QuantidadeServicos", "Cancelamento"]].head())

Dimensão da Gold: (5634, 28)


,id_cliente,DuracaoContrato,FaixaPermanencia,CobrancaMensal,RendaMensal,ComprometimentoRendaPct,QuantidadeServicos,Cancelamento
0,4578-PHJYZ,52,49–60 meses,68.75,6532,1.05,6,No
1,6289-CPNLD,33,25–36 meses,73.90,7634,0.97,6,Yes
2,2682-KEVRP,22,13–24 meses,20.05,3628,0.55,1,No
3,5697-GOMBF,28,25–36 meses,35.90,7851,0.46,2,No
4,9717-QEBGU,2,0–12 meses,70.30,1691,4.16,2,No


In [20]:
# Tabela geral de churn.
resumo_geral = pd.DataFrame({
    "Situação": ["Não cancelou", "Cancelou"],
    "Clientes": [int((df_gold["Cancelamento"] == "No").sum()), int((df_gold["Cancelamento"] == "Yes").sum())]
})
resumo_geral["Percentual da base"] = resumo_geral["Clientes"] / len(df_gold) * 100

total_clientes = len(df_gold)
total_cancelados = int(df_gold["CancelamentoBinario"].sum())
taxa_churn_geral = total_cancelados / total_clientes * 100

print(f"Clientes na base: {total_clientes:,}")
print(f"Clientes cancelados: {total_cancelados:,}")
print(f"Taxa geral de churn: {taxa_churn_geral:.2f}%")
display(resumo_geral)

Clientes na base: 5,634
Clientes cancelados: 1,504
Taxa geral de churn: 26.70%


,Situação,Clientes,Percentual da base
0,Não cancelou,4130,73.30
1,Cancelou,1504,26.70


## 5. Análise exploratória — perfil e comportamento de churn

### Pergunta 1 — Qual é a dimensão do cancelamento na carteira?

A taxa geral é a proporção de clientes cancelados em relação ao total de clientes. Nos próximos gráficos, sempre que a comparação for entre categorias (contrato, pagamento, serviços etc.), utilizaremos a **taxa dentro de cada categoria**, não somente a contagem de cancelados.

In [21]:
fig = px.pie(
    resumo_geral, names="Situação", values="Clientes", hole=0.58,
    color="Situação", color_discrete_map={"Não cancelou": "#2563EB", "Cancelou": "#DC2626"},
    title="Situação dos clientes na carteira", template="plotly_white"
)
fig.update_traces(textinfo="label+percent", hovertemplate="%{label}<br>Clientes: %{value:,}<extra></extra>")
fig.update_layout(title_x=0.5, legend_title_text="")
fig.show()

### Pergunta 2 — A modalidade contratual está associada ao cancelamento?

**Gráfico ajustado do notebook anterior:** em vez de mostrar apenas quantos clientes cancelaram em cada contrato, o gráfico apresenta a taxa de churn dentro de cada modalidade. Isso evita que um tipo de contrato pareça pior somente porque possui mais clientes.

In [22]:
gold_churn_contrato = (
    df_gold.groupby("Contrato", observed=True)
    .agg(Clientes=("id_cliente", "size"), Cancelamentos=("CancelamentoBinario", "sum"))
    .reset_index()
)
gold_churn_contrato["TaxaChurnPct"] = gold_churn_contrato["Cancelamentos"] / gold_churn_contrato["Clientes"] * 100
gold_churn_contrato = gold_churn_contrato.sort_values("TaxaChurnPct", ascending=True)
display(gold_churn_contrato)

fig = px.bar(
    gold_churn_contrato, x="TaxaChurnPct", y="Contrato", orientation="h",
    text=gold_churn_contrato["TaxaChurnPct"].map(lambda x: f"{x:.1f}%"),
    custom_data=["Clientes", "Cancelamentos"], color="TaxaChurnPct",
    color_continuous_scale=["#BFDBFE", "#1D4ED8"],
    title="Taxa de cancelamento por modalidade contratual",
    labels={"TaxaChurnPct": "Taxa de churn (%)", "Contrato": "Modalidade contratual"},
    template="plotly_white"
)
fig.update_traces(hovertemplate="<b>%{y}</b><br>Taxa de churn: %{x:.1f}%<br>Clientes: %{customdata[0]:,}<br>Cancelamentos: %{customdata[1]:,}<extra></extra>")
fig.update_layout(title_x=0.5, coloraxis_showscale=False, xaxis=dict(range=[0, max(50, gold_churn_contrato["TaxaChurnPct"].max()+5)], ticksuffix="%", gridcolor="#E5E7EB"), yaxis_title="")
fig.show()

,Contrato,Clientes,Cancelamentos,TaxaChurnPct
2,Two year,1347,44,3.27
1,One year,1188,138,11.62
0,Month-to-month,3099,1322,42.66


**Interpretação:** compare as taxas, e não apenas os volumes. Uma taxa maior em contratos mensais pode indicar que clientes sem compromisso de longo prazo são um grupo prioritário para ações de retenção. Isso não prova que o contrato, sozinho, cause o cancelamento.

### Pergunta 3 — Em que momento do relacionamento o cancelamento é mais frequente?

**Gráficos antigos de duração reorganizados:** os meses foram agrupados em faixas anuais. O gráfico apresenta a taxa de churn de cada faixa e o número de clientes que compõe o grupo no hover. Assim, evita a confusão de misturar meses isolados e anos de permanência.

In [23]:
gold_churn_permanencia = (
    df_gold.dropna(subset=["FaixaPermanencia"])
    .groupby("FaixaPermanencia", observed=False)
    .agg(Clientes=("id_cliente", "size"), Cancelamentos=("CancelamentoBinario", "sum"))
    .reset_index()
)
gold_churn_permanencia["TaxaChurnPct"] = gold_churn_permanencia["Cancelamentos"] / gold_churn_permanencia["Clientes"] * 100
display(gold_churn_permanencia)

fig = px.bar(
    gold_churn_permanencia, x="FaixaPermanencia", y="TaxaChurnPct",
    text=gold_churn_permanencia["TaxaChurnPct"].map(lambda x: f"{x:.1f}%"),
    custom_data=["Clientes", "Cancelamentos"], color="TaxaChurnPct",
    color_continuous_scale="Blues", title="Taxa de churn por faixa de permanência",
    labels={"FaixaPermanencia": "Tempo de relacionamento", "TaxaChurnPct": "Taxa de churn (%)"},
    template="plotly_white"
)
fig.update_traces(hovertemplate="<b>%{x}</b><br>Taxa de churn: %{y:.1f}%<br>Clientes no grupo: %{customdata[0]:,}<br>Cancelamentos: %{customdata[1]:,}<extra></extra>")
fig.update_layout(title_x=0.5, coloraxis_showscale=False, yaxis=dict(range=[0, max(50, gold_churn_permanencia["TaxaChurnPct"].max()+5)], ticksuffix="%", gridcolor="#E5E7EB"), xaxis_title="Faixa de permanência")
fig.show()

,FaixaPermanencia,Clientes,Cancelamentos,TaxaChurnPct
0,0–12 meses,1751,819,46.77
1,13–24 meses,805,229,28.45
2,25–36 meses,670,155,23.13
3,37–48 meses,619,125,20.19
4,49–60 meses,662,92,13.90
5,61–72 meses,1108,76,6.86


In [24]:
# Complemento: volume absoluto de cancelamentos por faixa de permanência.
fig = px.bar(
    gold_churn_permanencia, x="FaixaPermanencia", y="Cancelamentos",
    text="Cancelamentos", color="Cancelamentos", color_continuous_scale="Blues",
    title="Quantidade de cancelamentos por faixa de permanência",
    labels={"FaixaPermanencia": "Tempo de relacionamento", "Cancelamentos": "Clientes cancelados"},
    template="plotly_white"
)
fig.update_traces(textposition="outside", hovertemplate="%{x}<br>Cancelamentos: %{y:,}<extra></extra>")
fig.update_layout(title_x=0.5, coloraxis_showscale=False, yaxis=dict(rangemode="tozero", gridcolor="#E5E7EB"))
fig.show()

**Interpretação:** a taxa é mais adequada para comparar faixas com quantidades diferentes de clientes; o gráfico de volume responde quantos cancelamentos foram observados em cada faixa. Os dois indicadores respondem perguntas diferentes e devem ser lidos em conjunto.

### Pergunta 4 — Quais serviços apresentam maior taxa de cancelamento?

**Gráfico antigo de serviços ajustado:** a versão original contava serviços apenas entre quem cancelou. Isso não permitia saber se um serviço estava associado a maior risco. Agora comparamos clientes que possuem e não possuem cada serviço, calculando a taxa de churn em cada grupo.

In [25]:
condicoes_servicos = {
    "Telefone": df_gold["ServicoTelefonico"].eq("Yes"),
    "Múltiplas linhas": df_gold["MultiplasLinhas"].eq("Yes"),
    "Internet": df_gold["ServicoInternet"].isin(["DSL", "Fiber optic"]),
    "Segurança online": df_gold["SegurancaOnline"].eq("Yes"),
    "Backup online": df_gold["BackupOnline"].eq("Yes"),
    "Proteção do dispositivo": df_gold["ProtecaoDispositivo"].eq("Yes"),
    "Suporte técnico": df_gold["SuporteTecnico"].eq("Yes"),
    "Streaming de TV": df_gold["StreamingTV"].eq("Yes"),
    "Streaming de filmes": df_gold["StreamingFilmes"].eq("Yes"),
}

linhas_servicos = []
for nome_servico, possui in condicoes_servicos.items():
    for status, mascara in [("Possui", possui), ("Não possui", ~possui)]:
        grupo = df_gold.loc[mascara]
        n = len(grupo)
        cancelamentos = int(grupo["CancelamentoBinario"].sum())
        linhas_servicos.append({
            "Servico": nome_servico,
            "Situacao": status,
            "Clientes": n,
            "Cancelamentos": cancelamentos,
            "TaxaChurnPct": cancelamentos / n * 100 if n else np.nan
        })

gold_churn_servicos = pd.DataFrame(linhas_servicos)
display(gold_churn_servicos.sort_values(["Servico", "Situacao"]))

fig = px.bar(
    gold_churn_servicos, x="Servico", y="TaxaChurnPct", color="Situacao",
    barmode="group", custom_data=["Clientes", "Cancelamentos"],
    color_discrete_map={"Possui": "#2563EB", "Não possui": "#F59E0B"},
    title="Taxa de churn conforme a contratação de serviços",
    labels={"Servico": "Serviço", "TaxaChurnPct": "Taxa de churn (%)", "Situacao": "Situação do serviço"},
    template="plotly_white"
)
fig.update_traces(hovertemplate="<b>%{x}</b><br>%{fullData.name}<br>Taxa de churn: %{y:.1f}%<br>Clientes: %{customdata[0]:,}<br>Cancelamentos: %{customdata[1]:,}<extra></extra>")
fig.update_layout(title_x=0.5, yaxis=dict(ticksuffix="%", gridcolor="#E5E7EB"), xaxis_tickangle=-25, legend_title_text="")
fig.show()

,Servico,Situacao,Clientes,Cancelamentos,TaxaChurnPct
9,Backup online,Não possui,3690,1070,29.00
8,Backup online,Possui,1944,434,22.33
5,Internet,Não possui,1233,89,7.22
4,Internet,Possui,4401,1415,32.15
3,Múltiplas linhas,Não possui,3267,817,25.01
2,Múltiplas linhas,Possui,2367,687,29.02
11,Proteção do dispositivo,Não possui,3702,1052,28.42
10,Proteção do dispositivo,Possui,1932,452,23.40
7,Segurança online,Não possui,4036,1264,31.32
6,Segurança online,Possui,1598,240,15.02


### Pergunta 5 — A quantidade de serviços contratados está associada ao churn?

O gráfico original mostrava a participação de clientes cancelados com um ou mais serviços, mas não comparava com os clientes que permaneceram. Nesta versão, a taxa é calculada para cada quantidade de serviços na carteira inteira.

In [26]:
gold_churn_qtd_servicos = (
    df_gold.groupby("QuantidadeServicos", observed=True)
    .agg(Clientes=("id_cliente", "size"), Cancelamentos=("CancelamentoBinario", "sum"))
    .reset_index()
)
gold_churn_qtd_servicos["TaxaChurnPct"] = gold_churn_qtd_servicos["Cancelamentos"] / gold_churn_qtd_servicos["Clientes"] * 100
display(gold_churn_qtd_servicos)

fig = px.bar(
    gold_churn_qtd_servicos, x="QuantidadeServicos", y="TaxaChurnPct",
    text=gold_churn_qtd_servicos["TaxaChurnPct"].map(lambda x: f"{x:.1f}%"),
    custom_data=["Clientes", "Cancelamentos"], color="TaxaChurnPct", color_continuous_scale="Blues",
    title="Taxa de churn por quantidade de serviços contratados",
    labels={"QuantidadeServicos": "Quantidade de serviços", "TaxaChurnPct": "Taxa de churn (%)"},
    template="plotly_white"
)
fig.update_traces(hovertemplate="Serviços: %{x}<br>Taxa de churn: %{y:.1f}%<br>Clientes: %{customdata[0]:,}<br>Cancelamentos: %{customdata[1]:,}<extra></extra>")
fig.update_layout(title_x=0.5, coloraxis_showscale=False, yaxis=dict(ticksuffix="%", gridcolor="#E5E7EB"))
fig.show()

,QuantidadeServicos,Clientes,Cancelamentos,TaxaChurnPct
0,1,1021,112,10.97
1,2,682,202,29.62
2,3,692,305,44.08
3,4,765,280,36.60
4,5,737,247,33.51
5,6,720,183,25.42
6,7,538,122,22.68
7,8,321,42,13.08
8,9,158,11,6.96


### Pergunta 6 — O método de pagamento e o tipo de internet estão associados ao churn?

In [27]:
def tabela_taxa_por_categoria(df, coluna, nome_categoria):
    tabela = (
        df.groupby(coluna, observed=True)
        .agg(Clientes=("id_cliente", "size"), Cancelamentos=("CancelamentoBinario", "sum"))
        .reset_index()
        .rename(columns={coluna: nome_categoria})
    )
    tabela["TaxaChurnPct"] = tabela["Cancelamentos"] / tabela["Clientes"] * 100
    return tabela.sort_values("TaxaChurnPct", ascending=False)

gold_churn_pagamento = tabela_taxa_por_categoria(df_gold, "MetodoPagamento", "MetodoPagamento")
display(gold_churn_pagamento)

fig = px.bar(
    gold_churn_pagamento, x="TaxaChurnPct", y="MetodoPagamento", orientation="h",
    text=gold_churn_pagamento["TaxaChurnPct"].map(lambda x: f"{x:.1f}%"),
    custom_data=["Clientes", "Cancelamentos"], color="TaxaChurnPct", color_continuous_scale="Oranges",
    title="Taxa de churn por método de pagamento",
    labels={"TaxaChurnPct": "Taxa de churn (%)", "MetodoPagamento": "Método de pagamento"},
    template="plotly_white"
)
fig.update_traces(hovertemplate="<b>%{y}</b><br>Taxa de churn: %{x:.1f}%<br>Clientes: %{customdata[0]:,}<br>Cancelamentos: %{customdata[1]:,}<extra></extra>")
fig.update_layout(title_x=0.5, coloraxis_showscale=False, xaxis=dict(ticksuffix="%", gridcolor="#E5E7EB"), yaxis_title="")
fig.show()

,MetodoPagamento,Clientes,Cancelamentos,TaxaChurnPct
2,Electronic check,1888,852,45.13
3,Mailed check,1283,249,19.41
0,Bank transfer (automatic),1229,211,17.17
1,Credit card (automatic),1234,192,15.56


In [40]:
gold_churn_internet = tabela_taxa_por_categoria(df_gold, "ServicoInternet", "ServicoInternet")
display(gold_churn_internet)

fig = px.bar(
    gold_churn_internet, x="TaxaChurnPct", y="ServicoInternet", orientation="h",
    text=gold_churn_internet["TaxaChurnPct"].map(lambda x: f"{x:.1f}%"),
    custom_data=["Clientes", "Cancelamentos"], color="TaxaChurnPct", color_continuous_scale="Blues",
    title="Taxa de churn por modalidade de internet",
    labels={"TaxaChurnPct": "Taxa de churn (%)", "ServicoInternet": "Serviço de internet"},
    template="plotly_white"
)
fig.update_traces(hovertemplate="<b>%{y}</b><br>Taxa de churn: %{x:.1f}%<br>Clientes: %{customdata[0]:,}<br>Cancelamentos: %{customdata[1]:,}<extra></extra>")
fig.update_layout(title_x=0.5, coloraxis_showscale=False, xaxis=dict(ticksuffix="%", gridcolor="#8D9FC6"), yaxis_title="")
fig.show()

,ServicoInternet,Clientes,Cancelamentos,TaxaChurnPct
1,Fiber optic,2477,1042,42.07
0,DSL,1924,373,19.39
2,No,1233,89,7.22


## 6. Análise financeira — cobrança, renda e cancelamento

Esta seção responde à pergunta: **os valores de cobrança e a renda registrada apresentam diferenças entre clientes que cancelaram e os que permaneceram?**

Importante: `CobrancaMensal` representa a cobrança mensal; `CobrancaTotal` representa o acumulado; `RendaMensal` representa a renda mensal registrada. Não são medidas intercambiáveis. Além disso, a unidade monetária não está explicitada no dicionário fornecido.

### 6.1 Cobrança mensal por situação de cancelamento

O boxplot compara a distribuição das mensalidades. A linha central representa a mediana; a caixa representa o intervalo entre o primeiro e o terceiro quartil. Os pontos extremos ficam ocultos no gráfico para facilitar a leitura, mas os valores continuam na base.

In [29]:
resumo_cobranca_mensal = (
    df_gold.groupby("Cancelamento", observed=True)["CobrancaMensal"]
    .agg(Clientes="count", Media="mean", Mediana="median", Minimo="min", Maximo="max")
    .reset_index()
)
display(resumo_cobranca_mensal)

fig = px.box(
    df_gold, x="Cancelamento", y="CobrancaMensal", color="Cancelamento",
    points=False, category_orders={"Cancelamento": ["No", "Yes"]},
    color_discrete_map={"No": "#2563EB", "Yes": "#DC2626"},
    title="Distribuição da cobrança mensal por situação do cliente",
    labels={"Cancelamento": "Cancelou?", "CobrancaMensal": "Cobrança mensal (unidade da base)"},
    template="plotly_white"
)
fig.update_layout(title_x=0.5, showlegend=False, yaxis=dict(gridcolor="#E5E7EB"))
fig.show()

,Cancelamento,Clientes,Media,Mediana,Minimo,Maximo
0,No,4130,60.96,64.35,18.40,118.65
1,Yes,1504,74.63,79.82,18.85,118.35


### 6.2 Renda mensal por situação de cancelamento

A renda é analisada separadamente da cobrança. Uma diferença pequena entre grupos não deve ser interpretada como prova de que a renda não tenha qualquer relação com o churn; ela indica apenas que a comparação descritiva das distribuições pode não separar fortemente os grupos.

In [30]:
resumo_renda = (
    df_gold.groupby("Cancelamento", observed=True)["RendaMensal"]
    .agg(Clientes="count", Media="mean", Mediana="median", Minimo="min", Maximo="max")
    .reset_index()
)
display(resumo_renda)

fig = px.box(
    df_gold, x="Cancelamento", y="RendaMensal", color="Cancelamento", points=False,
    category_orders={"Cancelamento": ["No", "Yes"]},
    color_discrete_map={"No": "#2563EB", "Yes": "#DC2626"},
    title="Distribuição da renda mensal por situação do cliente",
    labels={"Cancelamento": "Cancelou?", "RendaMensal": "Renda mensal (unidade da base)"},
    template="plotly_white"
)
fig.update_layout(title_x=0.5, showlegend=False, yaxis=dict(gridcolor="#E5E7EB"))
fig.show()

,Cancelamento,Clientes,Media,Mediana,Minimo,Maximo
0,No,4130,"5,082.03","5,048.00",100,9993
1,Yes,1504,"5,167.63","5,214.50",102,9987


### 6.3 Comprometimento da renda por quartil

Fórmula do indicador:

\[
\text{Comprometimento da renda (\%)} = \frac{\text{Cobrança mensal}}{\text{Renda mensal}} \times 100
\]

Os clientes são divididos em quatro grupos de tamanho semelhante, do menor (Q1) para o maior (Q4) comprometimento. A taxa de churn de cada grupo é calculada sobre todos os clientes que pertencem ao quartil.

In [31]:
resumo_comprometimento = (
    df_gold.dropna(subset=["QuartilComprometimento"])
    .groupby("QuartilComprometimento", observed=False)
    .agg(
        Clientes=("id_cliente", "size"),
        Cancelamentos=("CancelamentoBinario", "sum"),
        MedianaComprometimento=("ComprometimentoRendaPct", "median"),
        MediaComprometimento=("ComprometimentoRendaPct", "mean")
    )
    .reset_index()
)
resumo_comprometimento["TaxaChurnPct"] = resumo_comprometimento["Cancelamentos"] / resumo_comprometimento["Clientes"] * 100
display(resumo_comprometimento)

fig = px.bar(
    resumo_comprometimento, x="QuartilComprometimento", y="TaxaChurnPct",
    text=resumo_comprometimento["TaxaChurnPct"].map(lambda x: f"{x:.1f}%"),
    custom_data=["Clientes", "Cancelamentos", "MedianaComprometimento"],
    color="TaxaChurnPct", color_continuous_scale="Purples",
    title="Taxa de churn por quartil de comprometimento da renda",
    labels={"QuartilComprometimento": "Comprometimento da renda", "TaxaChurnPct": "Taxa de churn (%)"},
    template="plotly_white"
)
fig.update_traces(hovertemplate="<b>%{x}</b><br>Taxa de churn: %{y:.1f}%<br>Clientes: %{customdata[0]:,}<br>Cancelamentos: %{customdata[1]:,}<br>Mediana do comprometimento: %{customdata[2]:.2f}%<extra></extra>")
fig.update_layout(title_x=0.5, coloraxis_showscale=False, yaxis=dict(ticksuffix="%", gridcolor="#E5E7EB"))
fig.show()

,QuartilComprometimento,Clientes,Cancelamentos,MedianaComprometimento,MediaComprometimento,TaxaChurnPct
0,Q1 — menor,1409,216,0.43,0.44,15.33
1,Q2,1408,419,1.01,1.01,29.76
2,Q3,1408,433,1.66,1.73,30.75
3,Q4 — maior,1409,436,4.51,8.20,30.94


**Cuidado com valores extremos:** uma renda mensal muito baixa pode gerar um percentual de comprometimento muito alto. Por isso, o notebook apresenta mediana e média por quartil e não remove automaticamente valores extremos. A base deve ser investigada antes de qualquer exclusão ou correção.

In [32]:
# Distribuição do comprometimento por situação do cliente e pontos extremos para inspeção.
resumo_comp_churn = (
    df_gold.groupby("Cancelamento", observed=True)["ComprometimentoRendaPct"]
    .agg(Clientes="count", Media="mean", Mediana="median", Q1=lambda s: s.quantile(.25), Q3=lambda s: s.quantile(.75), P95=lambda s: s.quantile(.95), Maximo="max")
    .reset_index()
)
display(resumo_comp_churn)

fig = px.box(
    df_gold, x="Cancelamento", y="ComprometimentoRendaPct", color="Cancelamento", points=False,
    category_orders={"Cancelamento": ["No", "Yes"]},
    color_discrete_map={"No": "#2563EB", "Yes": "#DC2626"},
    title="Distribuição do comprometimento da renda por situação do cliente",
    labels={"Cancelamento": "Cancelou?", "ComprometimentoRendaPct": "Cobrança mensal / renda mensal (%)"},
    template="plotly_white"
)
fig.update_layout(title_x=0.5, showlegend=False, yaxis=dict(gridcolor="#E5E7EB"))
fig.show()

,Cancelamento,Clientes,Media,Mediana,Q1,Q3,P95,Maximo
0,No,4130,2.67,1.20,0.65,2.32,9.33,101.50
1,Yes,1504,3.33,1.44,0.97,2.70,11.33,93.14


### 6.4 Renda mensal × cobrança mensal

O gráfico de dispersão mostra cada cliente como um ponto. A cor indica a situação de cancelamento. A relação visual ajuda a localizar concentrações e valores extremos, mas não substitui a análise do comprometimento e não demonstra causalidade.

In [33]:
fig = px.scatter(
    df_gold, x="RendaMensal", y="CobrancaMensal", color="Cancelamento",
    opacity=0.55, render_mode="webgl",
    color_discrete_map={"No": "#2563EB", "Yes": "#DC2626"},
    hover_data=["Contrato", "ServicoInternet", "DuracaoContrato", "ComprometimentoRendaPct"],
    title="Renda mensal e cobrança mensal dos clientes",
    labels={"RendaMensal": "Renda mensal (unidade da base)", "CobrancaMensal": "Cobrança mensal (unidade da base)", "Cancelamento": "Cancelou?"},
    template="plotly_white"
)
fig.update_layout(title_x=0.5, legend_title_text="Cancelou?", xaxis=dict(gridcolor="#E5E7EB"), yaxis=dict(gridcolor="#E5E7EB"))
fig.show()

### 6.5 Cobrança acumulada × tempo de relacionamento — gráfico simplificado

Este é o gráfico que substitui a visualização mais confusa de `TotalCharges` contra `tenure` em pontos individuais. Em vez de desenhar milhares de pontos sobrepostos, agrupamos os clientes em faixas de permanência e exibimos a **mediana da cobrança acumulada** para quem cancelou e para quem permaneceu.

A leitura fica mais direta: verificar como o acumulado cresce conforme o tempo de relacionamento e comparar os dois grupos. Como `CobrancaTotal` é acumulada, é esperado que clientes com mais meses apresentem valores maiores. A comparação deve sempre considerar a faixa de permanência.

In [34]:
base_total_tenure = df_gold.dropna(subset=["FaixaPermanencia", "CobrancaTotal"]).copy()
resumo_total_tenure = (
    base_total_tenure.groupby(["FaixaPermanencia", "Cancelamento"], observed=False)
    .agg(Clientes=("id_cliente", "size"), MedianaCobrancaTotal=("CobrancaTotal", "median"))
    .reset_index()
)
display(resumo_total_tenure)

fig = px.line(
    resumo_total_tenure, x="FaixaPermanencia", y="MedianaCobrancaTotal", color="Cancelamento",
    markers=True, custom_data=["Clientes"],
    category_orders={"FaixaPermanencia": faixas_tenure, "Cancelamento": ["No", "Yes"]},
    color_discrete_map={"No": "#2563EB", "Yes": "#DC2626"},
    title="Mediana da cobrança acumulada por faixa de permanência",
    labels={"FaixaPermanencia": "Tempo de relacionamento", "MedianaCobrancaTotal": "Mediana da cobrança acumulada (unidade da base)", "Cancelamento": "Cancelou?"},
    template="plotly_white"
)
fig.update_traces(hovertemplate="<b>%{x}</b><br>%{fullData.name}<br>Mediana acumulada: %{y:,.2f}<br>Clientes considerados: %{customdata[0]:,}<extra></extra>")
fig.update_layout(title_x=0.5, legend_title_text="Cancelou?", yaxis=dict(gridcolor="#E5E7EB"), xaxis_title="Faixa de permanência")
fig.show()

,FaixaPermanencia,Cancelamento,Clientes,MedianaCobrancaTotal
0,0–12 meses,No,891,177.40
1,0–12 meses,Yes,769,167.30
2,13–24 meses,No,555,966.25
3,13–24 meses,Yes,216,"1,367.05"
4,25–36 meses,No,492,"1,911.45"
5,25–36 meses,Yes,145,"2,596.15"
6,37–48 meses,No,474,"2,657.35"
7,37–48 meses,Yes,117,"3,706.95"
8,49–60 meses,No,547,"4,018.55"
9,49–60 meses,Yes,83,"5,036.30"


**Interpretação:** não compare a cobrança acumulada sem considerar o tempo de relacionamento. Um cliente que permaneceu por mais meses teve mais oportunidades de acumular cobranças. Os valores ausentes de `CobrancaTotal` foram excluídos somente deste gráfico e continuam identificados na Silver.

### 6.6 Resumo financeiro por situação de cancelamento

A tabela reúne quantidade de registros válidos, média e mediana de cobrança mensal, acumulada, renda e comprometimento. Ela facilita a preparação dos principais resultados para os slides.

In [35]:
linhas = []
for situacao, grupo in df_gold.groupby("Cancelamento", observed=True):
    linhas.append({
        "Cancelamento": situacao,
        "Clientes": len(grupo),
        "CobrancaMensal_media": grupo["CobrancaMensal"].mean(),
        "CobrancaMensal_mediana": grupo["CobrancaMensal"].median(),
        "CobrancaTotal_media": grupo["CobrancaTotal"].mean(),
        "CobrancaTotal_mediana": grupo["CobrancaTotal"].median(),
        "RendaMensal_media": grupo["RendaMensal"].mean(),
        "RendaMensal_mediana": grupo["RendaMensal"].median(),
        "Comprometimento_medio_pct": grupo["ComprometimentoRendaPct"].mean(),
        "Comprometimento_mediano_pct": grupo["ComprometimentoRendaPct"].median(),
        "Permanencia_media_meses": grupo["DuracaoContrato"].mean(),
        "Permanencia_mediana_meses": grupo["DuracaoContrato"].median(),
    })
resumo_financeiro = pd.DataFrame(linhas)
display(resumo_financeiro)

,Cancelamento,Clientes,CobrancaMensal_media,CobrancaMensal_mediana,CobrancaTotal_media,CobrancaTotal_mediana,RendaMensal_media,RendaMensal_mediana,Comprometimento_medio_pct,Comprometimento_mediano_pct,Permanencia_media_meses,Permanencia_mediana_meses
0,No,4130,60.96,64.35,"2,519.49","1,663.62","5,082.03","5,048.00",2.67,1.20,37.38,37.00
1,Yes,1504,74.63,79.82,"1,562.42",753.15,"5,167.63","5,214.50",3.33,1.44,18.33,10.00


## 7. Perfil dos clientes — variáveis complementares

As características demográficas e familiares ajudam a descrever a carteira. Não devem ser usadas isoladamente para rotular clientes como propensos ao cancelamento; a interpretação deve considerar quantidade de pessoas por grupo e o contexto de negócio.

In [36]:
colunas_perfil = [
    ("Senioridade", "Senioridade (0 = não; 1 = sim)"),
    ("Parceiro", "Possui parceiro(a)"),
    ("Dependentes", "Possui dependentes"),
    ("FaturaEletronica", "Fatura eletrônica"),
]
linhas_perfil = []
for coluna, rotulo in colunas_perfil:
    tabela = (
        df_gold.groupby(coluna, dropna=False, observed=True)
        .agg(Clientes=("id_cliente", "size"), Cancelamentos=("CancelamentoBinario", "sum"))
        .reset_index()
        .rename(columns={coluna: "Categoria"})
    )
    tabela["Variavel"] = rotulo
    tabela["TaxaChurnPct"] = tabela["Cancelamentos"] / tabela["Clientes"] * 100
    linhas_perfil.append(tabela)
resumo_perfil = pd.concat(linhas_perfil, ignore_index=True)
display(resumo_perfil[["Variavel", "Categoria", "Clientes", "Cancelamentos", "TaxaChurnPct"]])

fig = px.bar(
    resumo_perfil, x="Variavel", y="TaxaChurnPct", color="Categoria", barmode="group",
    custom_data=["Clientes", "Cancelamentos"],
    title="Taxa de churn por características do perfil dos clientes",
    labels={"Variavel": "Característica", "TaxaChurnPct": "Taxa de churn (%)", "Categoria": "Categoria"},
    template="plotly_white"
)
fig.update_traces(hovertemplate="<b>%{x}</b><br>Categoria: %{fullData.name}<br>Taxa: %{y:.1f}%<br>Clientes: %{customdata[0]:,}<br>Cancelamentos: %{customdata[1]:,}<extra></extra>")
fig.update_layout(title_x=0.5, yaxis=dict(ticksuffix="%", gridcolor="#E5E7EB"), xaxis_tickangle=-15)
fig.show()

,Variavel,Categoria,Clientes,Cancelamentos,TaxaChurnPct
0,Senioridade (0 = não; 1 = sim),0,4706,1111,23.61
1,Senioridade (0 = não; 1 = sim),1,928,393,42.35
2,Possui parceiro(a),No,2907,956,32.89
3,Possui parceiro(a),Yes,2727,548,20.10
4,Possui dependentes,No,3949,1246,31.55
5,Possui dependentes,Yes,1685,258,15.31
6,Fatura eletrônica,No,2307,383,16.60
7,Fatura eletrônica,Yes,3327,1121,33.69


## 8. Feedback dos clientes — análise textual complementar

A nuvem de palavras é mantida como recurso exploratório complementar ao notebook anterior. Ela mostra termos recorrentes nos comentários, separados por situação de cancelamento. A visualização não mede sentimento automaticamente, não identifica a causa do cancelamento e não substitui a leitura de amostras dos comentários.

In [37]:
try:
    from wordcloud import STOPWORDS, WordCloud
    import matplotlib.pyplot as plt

    palavras_ignoradas = set(STOPWORDS)
    palavras_ignoradas.update({"customer", "customers", "company", "provider", "overall", "experience", "also", "really"})

    texto_nao_cancelou = " ".join(
        df_gold.loc[df_gold["Cancelamento"].eq("No"), "FeedbackCliente"].dropna().astype(str)
    )
    texto_cancelou = " ".join(
        df_gold.loc[df_gold["Cancelamento"].eq("Yes"), "FeedbackCliente"].dropna().astype(str)
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for ax, texto, titulo, mapa in [
        (axes[0], texto_nao_cancelou, "Clientes que permaneceram", "Blues"),
        (axes[1], texto_cancelou, "Clientes que cancelaram", "Reds"),
    ]:
        if texto.strip():
            nuvem = WordCloud(width=800, height=600, background_color="white", stopwords=palavras_ignoradas,
                              max_words=35, collocations=False, colormap=mapa, random_state=42).generate(texto)
            ax.imshow(nuvem, interpolation="bilinear")
            ax.set_title(titulo, fontsize=14)
        else:
            ax.text(0.5, 0.5, "Sem comentários disponíveis", ha="center", va="center")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Biblioteca wordcloud não instalada. Instale com: pip install wordcloud")

Biblioteca wordcloud não instalada. Instale com: pip install wordcloud


## 9. Síntese dos resultados e respostas às perguntas de negócio

As células seguintes produzem um resumo numérico dos grupos com maiores taxas e destacam os indicadores financeiros. Usem esses resultados como base para a apresentação; revisem as interpretações considerando os tamanhos dos grupos e as limitações da base.

In [38]:
# Principais categorias com maiores taxas de churn.
contrato_top = gold_churn_contrato.sort_values("TaxaChurnPct", ascending=False).iloc[0]
permanencia_top = gold_churn_permanencia.sort_values("TaxaChurnPct", ascending=False).iloc[0]
pagamento_top = gold_churn_pagamento.sort_values("TaxaChurnPct", ascending=False).iloc[0]
internet_top = gold_churn_internet.sort_values("TaxaChurnPct", ascending=False).iloc[0]

# Comparação financeira: média dos grupos.
financeiro_por_status = resumo_financeiro.set_index("Cancelamento")
media_mensal_no = financeiro_por_status.loc["No", "CobrancaMensal_media"]
media_mensal_yes = financeiro_por_status.loc["Yes", "CobrancaMensal_media"]
media_renda_no = financeiro_por_status.loc["No", "RendaMensal_media"]
media_renda_yes = financeiro_por_status.loc["Yes", "RendaMensal_media"]

print(f"1) Taxa geral de churn: {taxa_churn_geral:.1f}% ({total_cancelados:,} de {total_clientes:,} clientes).")
print(f"2) Modalidade contratual com maior taxa: {contrato_top['Contrato']} ({contrato_top['TaxaChurnPct']:.1f}%; n={int(contrato_top['Clientes']):,}).")
print(f"3) Faixa de permanência com maior taxa: {permanencia_top['FaixaPermanencia']} ({permanencia_top['TaxaChurnPct']:.1f}%; n={int(permanencia_top['Clientes']):,}).")
print(f"4) Método de pagamento com maior taxa: {pagamento_top['MetodoPagamento']} ({pagamento_top['TaxaChurnPct']:.1f}%; n={int(pagamento_top['Clientes']):,}).")
print(f"5) Serviço de internet com maior taxa: {internet_top['ServicoInternet']} ({internet_top['TaxaChurnPct']:.1f}%; n={int(internet_top['Clientes']):,}).")
print(f"6) Cobrança mensal média — não cancelou: {media_mensal_no:.2f}; cancelou: {media_mensal_yes:.2f}.")
print(f"7) Renda mensal média — não cancelou: {media_renda_no:.2f}; cancelou: {media_renda_yes:.2f}.")

1) Taxa geral de churn: 26.7% (1,504 de 5,634 clientes).
2) Modalidade contratual com maior taxa: Month-to-month (42.7%; n=3,099).
3) Faixa de permanência com maior taxa: 0–12 meses (46.8%; n=1,751).
4) Método de pagamento com maior taxa: Electronic check (45.1%; n=1,888).
5) Serviço de internet com maior taxa: Fiber optic (42.1%; n=2,477).
6) Cobrança mensal média — não cancelou: 60.96; cancelou: 74.63.
7) Renda mensal média — não cancelou: 5082.03; cancelou: 5167.63.


### Interpretação consolidada

Ao finalizar a execução, preencham os pontos abaixo com os valores gerados:

- **Perfil e cancelamento:** a taxa geral indica a dimensão do problema na carteira. Os grupos com maior taxa precisam ser analisados junto ao número de clientes de cada grupo.
- **Contrato e permanência:** se a taxa for maior em contratos mensais e nas primeiras faixas de permanência, a empresa pode investigar ações de integração inicial, acompanhamento nos primeiros meses e benefícios de retenção.
- **Serviços:** taxas diferentes entre clientes com e sem um serviço podem indicar oportunidades para revisar qualidade, suporte ou percepção de valor. Não concluam que retirar ou adicionar um serviço reduzirá o churn sem testar essa hipótese.
- **Financeiro:** compare a cobrança mensal, renda e comprometimento. `CobrancaTotal` precisa ser lida em conjunto com a duração, pois é um valor acumulado.
- **Pagamento:** se um método apresentar taxa superior, investiguem fricção no processo de pagamento, falhas de cobrança e facilidade de migração para métodos automáticos.

### Possíveis recomendações para a Terra Signal
1. Desenvolver acompanhamento preventivo para clientes nos primeiros 12 meses, especialmente nos grupos com taxas mais elevadas.
2. Revisar a proposta de valor e a comunicação dos planos com mensalidades elevadas, oferecendo opções adequadas ao perfil de uso.
3. Investigar a experiência dos clientes que utilizam cheque eletrônico, verificando etapas de pagamento, falhas e conveniência.
4. Avaliar programas de fidelização e benefícios para migração voluntária para contratos de maior duração, sem impor barreiras ao cancelamento.
5. Investigar reclamações e comentários dos clientes para relacionar motivos textuais com serviços, cobrança e suporte.

As recomendações são hipóteses de ação derivadas da análise descritiva. A efetividade deve ser avaliada por testes controlados ou acompanhamento de indicadores após a implantação.

## 10. Exportação das tabelas Gold e encerramento

A Gold pode ter diferentes tabelas agregadas. Neste notebook, salvamos a base de clientes enriquecida e as principais tabelas de indicadores em CSV para facilitar a reutilização em SQL, dashboards e slides.

In [39]:
PASTA_GOLD = Path("gold")
PASTA_GOLD.mkdir(exist_ok=True)

tabelas_gold = {
    "gold_clientes.csv": df_gold,
    "gold_churn_contrato.csv": gold_churn_contrato,
    "gold_churn_permanencia.csv": gold_churn_permanencia,
    "gold_churn_servicos.csv": gold_churn_servicos,
    "gold_churn_qtd_servicos.csv": gold_churn_qtd_servicos,
    "gold_churn_pagamento.csv": gold_churn_pagamento,
    "gold_churn_internet.csv": gold_churn_internet,
    "gold_churn_comprometimento.csv": resumo_comprometimento,
    "gold_resumo_financeiro.csv": resumo_financeiro,
}

for nome_arquivo, tabela in tabelas_gold.items():
    tabela.to_csv(PASTA_GOLD / nome_arquivo, index=False, encoding="utf-8-sig")

print(f"{len(tabelas_gold)} arquivos Gold exportados para: {PASTA_GOLD.resolve()}")
print("Notebook concluído: Bronze preservada, Silver tratada e Gold preparada para análise.")

9 arquivos Gold exportados para: C:\Users\giova\OneDrive\Documentos\PII2_databricks\gold
Notebook concluído: Bronze preservada, Silver tratada e Gold preparada para análise.


## 11. Checklist final da entrega

- [ ] Contextualização do problema e perguntas de negócio apresentadas.
- [ ] Dicionário das 23 variáveis documentado.
- [ ] Camada Bronze preservada e diagnosticada.
- [ ] Tratamentos da Silver documentados e justificados.
- [ ] Ausências e duplicidades verificadas.
- [ ] Regras de negócio e integridade dos serviços validadas.
- [ ] Comparação Bronze × Silver apresentada.
- [ ] Arquivo `base_silver.csv` exportado e relido.
- [ ] Camada Gold e indicadores derivados construídos.
- [ ] Gráficos acompanhados de interpretações.
- [ ] Resultados e recomendações para a Terra Signal apresentados.
- [ ] Notebook executado do início ao fim, salvo com as saídas visíveis.

**Antes de enviar:** substitua `GX` e `Nome` no nome do arquivo pelo grupo e pelo seu nome, conforme a regra da disciplina. Garanta que `history.csv` acompanhe o notebook caso o professor precise reproduzir a execução.